In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
uploaded = files.upload()

video_name = list(uploaded.keys())[0]
print("Uploaded video:", video_name)

In [ ]:
cap = cv2.VideoCapture(video_name)

if not cap.isOpened():
    print("Error: Could not open the video.")
else:
    print("Video opened successfully.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Width:", width)
print("Height:", height)

In [ ]:
ret, frame1 = cap.read()
ret2, frame2 = cap.read()

if not ret or not ret2:
    print("Error: Could not read video frames.")
else:
    print("Two consecutive frames loaded successfully.")

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
plt.title("Frame 1")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
plt.title("Frame 2")
plt.axis("off")

plt.show()

In [ ]:
# Parameters for Shi-Tomasi corner detection
feature_params = dict(
    maxCorners=100,
    qualityLevel=0.3,
    minDistance=7,
    blockSize=7
)

# Parameters for Lucas-Kanade Optical Flow
lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
              10, 0.03)
)

# Detect feature points in first frame
p0 = cv2.goodFeaturesToTrack(gray1, mask=None, **feature_params)

if p0 is None:
    print("No feature points detected.")
else:
    print("Number of feature points:", len(p0))

# Calculate optical flow
p1, st, err = cv2.calcOpticalFlowPyrLK(
    gray1,
    gray2,
    p0,
    None,
    **lk_params
)

# Select successfully tracked points
if p1 is not None:
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    print("Successfully tracked points:", len(good_new))

In [ ]:
# Create a copy of the second frame
lk_result = frame2.copy()

# Draw motion vectors
for new, old in zip(good_new, good_old):
    x_new, y_new = new.ravel()
    x_old, y_old = old.ravel()

    cv2.arrowedLine(
        lk_result,
        (int(x_old), int(y_old)),
        (int(x_new), int(y_new)),
        (0, 255, 0),
        2,
        tipLength=0.3
    )

    cv2.circle(
        lk_result,
        (int(x_new), int(y_new)),
        4,
        (0, 0, 255),
        -1
    )

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(lk_result, cv2.COLOR_BGR2RGB))
plt.title("Lucas-Kanade Sparse Optical Flow")
plt.axis("off")
plt.show()


In [ ]:
flow = cv2.calcOpticalFlowFarneback(
    gray1,
    gray2,
    None,
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0
)

print("Farneback Dense Optical Flow calculated successfully.")

In [ ]:
# Convert flow vectors from Cartesian coordinates to polar coordinates
magnitude, angle = cv2.cartToPolar(
    flow[..., 0],
    flow[..., 1]
)

# Create HSV image
hsv = np.zeros_like(frame1)

# Hue represents direction
hsv[..., 0] = angle * 180 / np.pi / 2

# Saturation fixed
hsv[..., 1] = 255

# Value represents magnitude
hsv[..., 2] = cv2.normalize(
    magnitude,
    None,
    0,
    255,
    cv2.NORM_MINMAX
)

# Convert HSV to BGR
dense_result = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(dense_result, cv2.COLOR_BGR2RGB))
plt.title("Farneback Dense Optical Flow")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(frame2, cv2.COLOR_BGR2RGB))
plt.title("Original Frame")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(cv2.cvtColor(lk_result, cv2.COLOR_BGR2RGB))
plt.title("Lucas-Kanade Sparse Flow")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(cv2.cvtColor(dense_result, cv2.COLOR_BGR2RGB))
plt.title("Farneback Dense Flow")
plt.axis("off")

plt.show()

**Ques 1. What is Optical Flow? How is it used in computer vision?**

**Ans.** Optical Flow is a computer vision technique used to estimate the apparent motion of objects, surfaces, or the camera between two consecutive video frames. It analyzes changes in image pixels to determine the direction and amount of motion. Optical flow is used in applications such as object tracking, autonomous driving, video surveillance, robotics, gesture recognition, sports analytics, and augmented reality.

**Ques 2. Explain the working principle of the Lucas-Kanade Optical Flow algorithm.**

**Ans.** Lucas-Kanade is a Sparse Optical Flow algorithm that estimates the motion of selected feature points between consecutive frames. First, important feature points such as corners are detected in the first frame. The algorithm then assumes that the pixel intensity of a point remains approximately constant while moving and that nearby pixels have similar motion. Using these assumptions, it calculates the displacement of the selected points in the next frame and represents their movement using motion vectors or trajectories.

**Ques 3. What is Dense Optical Flow? How does it differ from Sparse Optical Flow?**

**Ans.** Dense Optical Flow estimates the motion for almost every pixel in an image or video frame. In contrast, Sparse Optical Flow calculates motion only for selected feature points. Dense optical flow provides a complete motion field and is useful for detailed motion analysis, while sparse optical flow requires less computation and is useful for tracking important points or objects.

**Ques 4. Compare the Lucas-Kanade and Farneback optical flow algorithms.**

**Ans.** Lucas-Kanade is a Sparse Optical Flow algorithm that tracks selected feature points, making it computationally efficient and suitable for object or feature tracking. Farneback is a Dense Optical Flow algorithm that estimates motion across the image and therefore provides more complete motion information. Lucas-Kanade generally has lower computational complexity, whereas Farneback provides more detailed information about motion throughout the scene.

**Ques 5. What assumptions are made while computing optical flow?**

**Ans.** Optical flow computation is based on several important assumptions. The first is the brightness or intensity constancy assumption, which means that the appearance of a moving point remains approximately the same between consecutive frames. The second is the assumption of small motion between frames. Another assumption is that neighboring pixels generally have similar motion. These assumptions allow the algorithm to estimate the movement of image points.

**Ques 6. What factors can affect the accuracy of optical flow estimation?**

**Ans.** The accuracy of optical flow can be affected by object speed, lighting conditions, camera motion, image noise, occlusion, textureless regions, and large displacement between consecutive frames. Very fast motion can make point correspondence difficult, while sudden changes in illumination can produce incorrect motion estimates. Camera movement can also cause many background points to appear in motion.

**Ques 7. Mention five real-world applications of optical flow in computer vision.**

**Ans.** Five important applications of optical flow are:

1. Autonomous driving
2. Video surveillance
3. Object tracking
4. Gesture recognition
5. Sports analytics

It is also useful in robotics, augmented reality, and action recognition.

**Ques 8. Why are grayscale images generally used for optical flow computation?**

**Ans.** Grayscale images are generally used because optical flow mainly depends on changes in image intensity rather than color information. Converting an image to grayscale reduces the amount of data that must be processed and simplifies calculations. It also makes the computation of image gradients and motion estimation more efficient.

**Ques 9. What are the limitations of optical flow algorithms in real-world environments?**

**Ans.** Optical flow algorithms have several limitations. Their accuracy can decrease when there are sudden lighting changes, image noise, occlusion, textureless regions, very fast motion, or large displacement between frames. Camera movement can also make motion estimation more difficult because both the background and objects may appear to move. Therefore, optical flow may not always provide reliable results in complex real-world environments.

**Ques 10. How does optical flow contribute to applications such as autonomous driving, video surveillance, and action recognition?**

**Ans.** Optical flow provides information about how objects and the camera are moving over time. In autonomous driving, it can help detect moving vehicles, pedestrians, and changes in the surrounding environment. In video surveillance, it can help identify movement and track objects or unusual activities. In action recognition, patterns of motion between consecutive frames can be analyzed to recognize human activities and gestures. Therefore, optical flow is important for understanding dynamic scenes in computer vision.
